# Norwegian fisheries seasonality — Gold consumer (A10)

**Question:** How do landed fish quantity and species composition vary by Norwegian
region and month?

This notebook reads only the **published** Gold mart (the version that passed the
validation in `src/gold/validate.py`), and answers the question with **Spark SQL**.
Quantity is **round weight in tonnes**: the source's live-weight-equivalent kilograms
divided by 1,000.

Run it from the repo root or the `notebooks/` folder. It uses the storage of
`PIPELINE_ENV` (default `dev`); set `PIPELINE_STORAGE_ROOT` to read the VM's data.

## 1. Open the published Gold

In [ ]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)  # config paths like ./data/dev are relative to the repo root

from common.config import load_config
from common.contracts import RunContext
from common.spark import get_spark
from gold.publish import published, published_tables

config = load_config(os.environ.get("PIPELINE_ENV", "dev"))
ctx = RunContext("notebook", config["env"], datetime.now(timezone.utc), config)
spark = get_spark(config)

current = published(ctx)  # (printing the full path would show your local folder names)
print("Published Gold from run", current["run_id"], "at", current["published_at"])

# Register each Gold table as a SQL view, so the queries below can use its name
for name, path in published_tables(ctx).items():
    spark.read.parquet(path).createOrReplaceTempView(name)

spark.sql("SHOW VIEWS").show(truncate=False)

## 2. The consumer view

One SQL query joins the fact table with its three dimensions (the star schema) and
applies the filters. `:year`, `:country`, `:county` and `:species` are **parameters**:
Spark fills them in safely, instead of building the SQL text by hand.
Leave `COUNTY_CODE` / `SPECIES_CODE` empty to include all.

In [ ]:
LANDING_YEAR = 2023
COUNTRY_CODE = "NOR"
COUNTY_CODE = ""    # Example: "54" for Troms og Finnmark
SPECIES_CODE = ""   # Example: "0201001" for cod

selected = spark.sql(
    """
    SELECT m.month_id, m.month, r.landing_county_code, r.landing_county_name,
           s.species_code, s.species_name,
           f.line_count, f.total_round_weight_kg,
           f.missing_round_weight_rows, f.zero_round_weight_rows
    FROM fact_monthly_species_region AS f
    JOIN dim_month AS m          USING (month_id)
    JOIN dim_landing_region AS r USING (region_id)
    JOIN dim_species AS s        USING (species_id)
    WHERE m.year = :year
      AND r.landing_country_code = :country
      AND (:county = '' OR r.landing_county_code = :county)
      AND (:species = '' OR s.species_code = :species)
    """,
    args={
        "year": LANDING_YEAR,
        "country": COUNTRY_CODE,
        "county": COUNTY_CODE,
        "species": SPECIES_CODE,
    },
)
selected.createOrReplaceTempView("selected")

## 3. Coverage and headline result

In [ ]:
spark.sql("""
    SELECT ROUND(SUM(total_round_weight_kg) / 1000, 3) AS round_weight_tonnes,
           SUM(line_count)                             AS source_lines,
           SUM(missing_round_weight_rows)              AS missing_round_weight,
           SUM(zero_round_weight_rows)                 AS zero_round_weight
    FROM selected
""").show(truncate=False)

## 4. Seasonal pattern

In [ ]:
spark.sql("""
    SELECT month_id,
           ROUND(SUM(total_round_weight_kg) / 1000, 3) AS round_weight_tonnes,
           SUM(line_count)                             AS source_lines
    FROM selected
    GROUP BY month_id
    ORDER BY month_id
""").show(12, truncate=False)

## 5. Largest species by round weight

In [ ]:
spark.sql("""
    SELECT species_code, species_name,
           ROUND(SUM(total_round_weight_kg) / 1000, 3) AS round_weight_tonnes,
           SUM(line_count)                             AS source_lines
    FROM selected
    GROUP BY species_code, species_name
    ORDER BY round_weight_tonnes DESC
    LIMIT 10
""").show(truncate=False)

## 6. Largest landing counties by round weight

Unknown county stays visible as its own row instead of being dropped.

In [ ]:
spark.sql("""
    SELECT landing_county_code, landing_county_name,
           ROUND(SUM(total_round_weight_kg) / 1000, 3) AS round_weight_tonnes,
           SUM(line_count)                             AS source_lines
    FROM selected
    GROUP BY landing_county_code, landing_county_name
    ORDER BY round_weight_tonnes DESC
    LIMIT 15
""").show(truncate=False)

## 7. Species composition: top 3 species in each month

A **window function** ranks the species inside each month by round weight, so we can
see how the species mix changes through the year.

In [ ]:
spark.sql("""
    WITH monthly AS (
        SELECT month_id, species_name, SUM(total_round_weight_kg) AS round_weight_kg
        FROM selected
        GROUP BY month_id, species_name
    ),
    ranked AS (
        SELECT month_id, species_name, round_weight_kg,
               round_weight_kg / SUM(round_weight_kg) OVER (PARTITION BY month_id) AS share,
               RANK() OVER (PARTITION BY month_id ORDER BY round_weight_kg DESC) AS rank
        FROM monthly
    )
    SELECT month_id, rank, species_name,
           ROUND(round_weight_kg / 1000, 3) AS round_weight_tonnes,
           ROUND(share * 100, 1)            AS percent_of_month
    FROM ranked
    WHERE rank <= 3
    ORDER BY month_id, rank
""").show(36, truncate=False)

## 8. Answer and limitations

*Write the answer from the results above* (which months peak, which species dominate
and how the mix changes through the year, which counties land the most).

Interpretation limits:

- A source row is a document product line, not one delivery or one fish.
- Round weight is the selected live-weight-equivalent measure; it must not be added to
  product or gross weight.
- Missing county and round weight stay explicit; zero round weight is kept and flagged
  by the Silver quality rules.

In [ ]:
spark.stop()